# FraudShield TRL Colab Demo

Minimal Colab training scaffold for the FraudShield FraudOps environment.

Default base model: `Qwen/Qwen2.5-0.5B-Instruct`


In [ ]:
%pip install -q trl transformers accelerate datasets peft openai openenv-core
!git clone https://github.com/DevikaJ2005/Fraudshield.git
%cd Fraudshield/fraudshield
%pip install -q -e .


In [ ]:
from fraudshield_env import FraudShieldEnvironment
from llm_agent import WorkflowHeuristicFraudOpsAgent
from graders import FraudShieldGrader
from models import FraudCheckAction

def rollout(agent, task_name):
    env = FraudShieldEnvironment(data_path='data', seed=42)
    env.load_data()
    obs = env.reset(task_name).observation
    while not env.is_done:
        action = agent.decide(obs)
        obs = env.step(action).observation
    return env.get_episode_report()

baseline_agent = WorkflowHeuristicFraudOpsAgent()
easy_summary = rollout(baseline_agent, 'easy')
medium_summary = rollout(baseline_agent, 'medium')
print('baseline easy', easy_summary['metrics'])
print('baseline medium', medium_summary['metrics'])


In [ ]:
import json
from datasets import Dataset

def collect_prompts(task_name, max_samples=16):
    env = FraudShieldEnvironment(data_path='data', seed=42)
    env.load_data()
    obs = env.reset(task_name).observation
    agent = WorkflowHeuristicFraudOpsAgent()
    samples = []
    while not env.is_done and len(samples) < max_samples:
        samples.append({
            'prompt': json.dumps({
                'case_id': obs.case_id,
                'task_name': obs.task_name.value,
                'current_screen': obs.current_screen.value,
                'visible_panels': obs.visible_panels,
                'revealed_evidence': obs.revealed_evidence,
                'linked_case_ids': obs.linked_case_ids,
                'remaining_steps': obs.remaining_steps,
                'remaining_sla': obs.remaining_sla,
                'note_required': obs.note_required,
                'allowed_actions': [a.value for a in obs.allowed_actions],
                'case_summary': obs.case_summary.model_dump(mode='json')
            }, separators=(',', ':'))
        })
        obs = env.step(agent.decide(obs)).observation
    return samples

train_dataset = Dataset.from_list(collect_prompts('easy') + collect_prompts('medium'))
train_dataset


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

SYSTEM_PROMPT = (
    'You are a fraud operations analyst. Return JSON with case_id, action_type, note_text, resolution, reasoning. '
    'Use only the allowed action names shown in the observation.'
)

def format_prompt(example):
    return {
        'prompt': f"{SYSTEM_PROMPT}\nObservation: {example['prompt']}\nRespond with JSON only."
    }

trl_dataset = train_dataset.map(format_prompt)

def reward_fn(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion[0]['content'] if isinstance(completion, list) else str(completion)
        try:
            start = text.find('{')
            end = text.rfind('}')
            payload = json.loads(text[start:end+1])
            action = FraudCheckAction.model_validate(payload)
            score = 1.0
            if action.action_type.value not in text:
                score -= 0.25
        except Exception:
            score = 0.0
        rewards.append(score)
    return rewards

config = GRPOConfig(
    output_dir='fraudshield-grpo-demo',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-6,
    max_prompt_length=2048,
    max_completion_length=160,
    num_generations=2,
    logging_steps=1,
    num_train_epochs=1,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_fn,
    args=config,
    train_dataset=trl_dataset,
    processing_class=tokenizer,
)

# trainer.train()  # Uncomment for a smoke run in Colab


In [ ]:
# After training, re-run easy/medium rollouts with your decoding wrapper and
# compare the before/after scores for the blog post.
# Target: >= +0.10 absolute improvement over the baseline on held-out states.
